# Colab setup (run first)

On **Google Colab**, run the two cells below first (mount Drive + install deps), then select a **GPU runtime** (Runtime -> Change runtime type -> GPU). Edit `DRIVE_PROJECT_PATH` to match where you uploaded the project. On a local machine these cells are skipped automatically.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_PROJECT_PATH = "/content/drive/MyDrive/PRAgenticAI"  # edit to your upload location

    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")

    PROJECT_ROOT = Path(DRIVE_PROJECT_PATH).resolve()
    assert PROJECT_ROOT.exists(), (
        f"Project folder not found at {PROJECT_ROOT}. Upload the project to Drive "
        "and set DRIVE_PROJECT_PATH to its location."
    )
    os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "requirements.txt").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    PROJECT_ROOT = PROJECT_ROOT.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab only). Colab already ships CUDA-enabled PyTorch.
if IN_COLAB:
    %pip install -q \
        "transformers>=4.46.0" \
        "datasets>=2.18.0" \
        "peft>=0.13.0" \
        "accelerate>=0.34.0" \
        sentencepiece \
        "evaluate>=0.4.0" \
        "sacrebleu>=2.4.0" \
        "bert-score>=0.3.13" \
        "codebleu>=0.7.0" \
        "nltk>=3.8.0" \
        "tree-sitter>=0.23.2,<0.24.0" \
        "tree-sitter-python>=0.23.6,<0.24.0" \
        "tree-sitter-java>=0.23.5,<0.24.0" \
        pyyaml
    # If CodeBLEU returns "an integer is required", re-run this cell and restart
    # the runtime (Runtime -> Restart session), then re-run from the setup cell.
    # Upgrading transformers/peft over Colab's preinstalled versions usually
    # REQUIRES a runtime restart before the new versions are imported. Restart
    # manually (Runtime -> Restart session) OR uncomment the line below to
    # restart automatically, then re-run from the setup cell at the top.
    # import os; os.kill(os.getpid(), 9)
    print("Dependencies installed.")
    print("IMPORTANT: If imports fail or you see a 'RESTART REQUIRED' notice, "
          "restart the runtime (Runtime -> Restart session) and re-run from the setup cell.")
else:
    print("Not on Colab; skipping pip install.")

# Baseline Qwen2.5-Coder-0.5B-Instruct — Step-by-Step Eval

Evaluate the **pre–fine-tune** base model on AVATAR-TC Java→Python pairs.

Metrics: **BLEU**, **BERTScore**, **CodeBLEU**, **CodeBERTScore** (all via `evaluation/metrics.py`).

Each step calls the shared functions in `evaluation/run_baseline_qwen.py` and `evaluation/metrics.py` — no logic is duplicated here. Run the cells top to bottom.

Set `SHUFFLE=True` and matching `SEED` in both notebooks so eval does not always use the first file-order samples (e.g. `calculateSquareSum`). Use the same `SHUFFLE` / `SEED` / `MAX_SAMPLES` when comparing against the fine-tuned notebook.

## 1. Imports & path setup

In [ ]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT  # type: ignore[used-before-def]  # noqa: F821
except NameError:
    _here = Path.cwd()
    PROJECT_ROOT = (_here if (_here / "requirements.txt").exists() else _here.parent).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inference.generator import CodeGenerator
from evaluation.run_baseline_qwen import (
    load_avatar_tc,
    generate_predictions,
    pick_inspect_index,
    compute_oop_breakdown,
)
from evaluation.metrics import (
    compute_bleu,
    compute_bert_score,
    compute_codebleu,
    compute_code_bert_score,
)

print("Project root:", PROJECT_ROOT)

## 2. Config

Start with a small `MAX_SAMPLES` smoke test, then set it to `None` for the full split.
Set `SKIP_BERT = True` to skip the (slower, model-downloading) BERTScore / CodeBERTScore steps while iterating.
Use the same `SHUFFLE` / `SEED` / `MAX_SAMPLES` as the fine-tuned notebook for a fair comparison.

In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
SPLIT = "valid"        # "valid" (443) or "test" (1745)
MAX_SAMPLES = 10        # None for full split
BATCH_SIZE = 8
DEVICE = "auto"         # auto -> CUDA on Colab, MPS on Mac, CPU otherwise
SKIP_BERT = False       # True to skip BERTScore / CodeBERTScore
SHUFFLE = True          # randomize which pairs are evaluated when MAX_SAMPLES is set
SEED = 42               # eval subset reproducibility (baseline vs fine-tuned); None = new random batch each run
INSPECT_SEED = None     # preview randomness only; SEED stays fixed for fair eval comparison
INSPECT_INDEX = None    # pin a sample index (overrides INSPECT_SEED), e.g. 3

## 3. Load AVATAR-TC data

In [ ]:
data = load_avatar_tc(SPLIT, MAX_SAMPLES, shuffle=SHUFFLE, seed=SEED)
print(f"Loaded {len(data)} samples from AVATAR-TC ({SPLIT})")
print(f"  shuffle={SHUFFLE}, seed={SEED}, unique programs={len({r['java_code'] for r in data})}\n")

PREVIEW_INDEX = (
    INSPECT_INDEX
    if INSPECT_INDEX is not None
    else pick_inspect_index(len(data), seed=INSPECT_SEED)
)

for idx, row in enumerate(data):
    tag = row["java_code"][:70].replace("\n", " ")
    mark = " <-- preview" if idx == PREVIEW_INDEX else ""
    print(f"  [{idx}] {tag}{mark}")

print(f"\n--- Java input (sample {PREVIEW_INDEX}) ---")
print(data[PREVIEW_INDEX]["java_code"][:400])
print(f"\n--- Python reference (sample {PREVIEW_INDEX}) ---")
print(data[PREVIEW_INDEX]["python_code"][:400])

## 4. Load the model

First run downloads the model weights from HuggingFace.

In [ ]:
gen = CodeGenerator(model_path=MODEL, device=DEVICE)

## 5. Generate Python translations

`max_new_tokens` is sized per-sample from the reference length; progress prints every `BATCH_SIZE` samples.

In [ ]:
import time

t0 = time.time()
predictions = generate_predictions(data, gen, batch_size=BATCH_SIZE)
gen_time = time.time() - t0

references = [item["python_code"] for item in data]
inputs = [item["java_code"] for item in data]

print(f"\nGeneration: {gen_time:.1f}s ({gen_time / len(data):.2f}s/sample)")

## 6. BLEU

In [ ]:
bleu = compute_bleu(predictions, references)
bleu

## 7. CodeBLEU

In [ ]:
codebleu = compute_codebleu(predictions, references)
codebleu

## 8. BERTScore (roberta-large)

Downloads `roberta-large` on first run. Skipped when `SKIP_BERT = True`.

In [ ]:
if SKIP_BERT:
    bertscore = {"bertscore_f1": None}
    print("Skipped (SKIP_BERT=True)")
else:
    bertscore = compute_bert_score(predictions, references)
bertscore

## 9. CodeBERTScore (microsoft/codebert-base)

Downloads `microsoft/codebert-base` on first run. Skipped when `SKIP_BERT = True`.

In [ ]:
if SKIP_BERT:
    code_bertscore = {"code_bertscore_f1": None}
    print("Skipped (SKIP_BERT=True)")
else:
    code_bertscore = compute_code_bert_score(predictions, references)
code_bertscore

## 10. Combined metrics table

In [ ]:
import pandas as pd

all_metrics = {**bleu, **codebleu, **bertscore, **code_bertscore}

metric_keys = [
    "bleu",
    "bertscore_f1",
    "codebleu",
    "code_bertscore_f1",
    "codebleu_ngram",
    "codebleu_weighted_ngram",
    "codebleu_syntax",
    "codebleu_dataflow",
]
summary = pd.Series(
    {k: all_metrics[k] for k in metric_keys if all_metrics.get(k) is not None}
).round(4)
summary

## 11. OOP category breakdown (optional)

Per-OOP-category CodeBLEU to spot weak spots (inheritance, interfaces, generics, ...).

In [ ]:
oop_breakdown = compute_oop_breakdown(predictions, references, inputs)
pd.DataFrame(oop_breakdown).T

## 12. Inspect a prediction vs reference

In [ ]:
i = PREVIEW_INDEX
print(f"=== Sample {i} ===")
print("=== Java input ===")
print(inputs[i][:500])
print("\n=== Prediction ===")
print(predictions[i][:500])
print("\n=== Reference ===")
print(references[i][:500])